# Attention U-Net segmentation of LC25000 trained on automatically generated masks

This notebook runs every experiment of the revised JTIE manuscript with the `lcseg` package of this repository
and writes all numbers of Tables 5–11 to `outputs/report/results.md`. **Numbers in the manuscript must be copied
from that file only.**

| Stage | Manuscript |
|---|---|
| `prepare` – resize to 256×256, automatic masks, coverage rule | Section III.E, Table 3 |
| `group` – source-tile grouping, source-disjoint split, leakage audit | Section III.C |
| `generator` – mask generator applied without class labels, timing | Section IV.C, Tables 7 and 9 |
| `launch` – training + test evaluation of every (variant, seed) run | Sections III.F–III.H, Tables 4, 5, 6, 8, 10, 11 |
| `bench` – parameters, GFLOPs, latency, memory | Section IV.E, Table 9 |
| `report` – all tables and figures | Section IV |

**Kaggle settings:** Accelerator *GPU T4 ×2*, Internet *on* (to clone the repository), and *Add Input* →
`andrewmvd/lung-and-colon-cancer-histopathological-images`.

**Several sessions:** the six training runs take longer than one 12-hour session if the GPUs are slow.
Every stage is resumable. After a session ends, use *Save Version*, add that version's output as an input
of the next session, put its path in `PREV`, and run all cells again; finished stages and runs are skipped.

In [ ]:
import os, sys, glob, shutil, subprocess, json

REPO_URL = "https://github.com/USERNAME/lc25000-attention-unet.git"   # <- change USERNAME
REPO = "/kaggle/working/lc25000-attention-unet"
OUT = "/kaggle/working/outputs"
PREV = []          # e.g. ["/kaggle/input/lcseg-session-1/outputs"] to continue from an earlier session
HOURS = 11.0       # do not start a new training run after this many hours
CONFIG = None      # optional JSON file that overrides lcseg/config.py (leave None for the manuscript settings)

In [ ]:
# get the code: from GitHub, or from a Kaggle dataset that contains the repository
if not os.path.exists(REPO):
    local = glob.glob("/kaggle/input/**/lcseg/__init__.py", recursive=True)
    if local:
        shutil.copytree(os.path.dirname(os.path.dirname(local[0])), REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
try:
    import albumentations
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=%H %cd"], capture_output=True, text=True).stdout)

In [ ]:
def find_lc25000(root="/kaggle/input"):
    # the lcseg package searches this folder recursively for the five class folders
    for d, dn, fn in os.walk(root, followlinks=True):
        if os.path.basename(d) == "lung_colon_image_set":
            return d
    return root

LC_ROOT = find_lc25000()
print("LC25000:", LC_ROOT)
os.makedirs(OUT, exist_ok=True)

# continue from earlier sessions
for p in PREV:
    for sub in ["cache", "groups", "generator", "bench"]:
        if os.path.exists(os.path.join(p, sub)) and not os.path.exists(os.path.join(OUT, sub)):
            shutil.copytree(os.path.join(p, sub), os.path.join(OUT, sub))
    for r in glob.glob(os.path.join(p, "runs", "*")):
        dst = os.path.join(OUT, "runs", os.path.basename(r))
        if os.path.exists(os.path.join(r, "eval.npz")) and not os.path.exists(dst):
            shutil.copytree(r, dst)

def lcseg(*args):
    cmd = [sys.executable, "-m", "lcseg", *args, "--out", OUT] + (["--config", CONFIG] if CONFIG else [])
    print(">>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

## 1. Masks (Section III.E) and source-disjoint split (Section III.C)

In [ ]:
lcseg("prepare", "--lc-root", LC_ROOT)
print(json.load(open(f"{OUT}/cache/prepare.json")))
lcseg("group")
lcseg("generator")

## 2. Training and test evaluation of all runs (one worker per GPU)
Progress is written to `outputs/worker_gpu*.log`.

In [ ]:
lcseg("launch", "--hours", str(HOURS))

In [ ]:
for log in sorted(glob.glob(f"{OUT}/worker_gpu*.log")):
    print("====", log)
    print("".join(l for l in open(log).readlines() if l.startswith("[")) [-4000:])

## 3. Computational cost (Table 9) and report (Tables 5–11, figures)

In [ ]:
lcseg("bench")
lcseg("report")
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{OUT}/report/results.md").read()))
for f in ["fig2_masks.png", "fig4_curves.png", "fig5_examples.png", "fig7_failure_cases.png",
          "fig_per_image_dice.png", "fig_ablation.png"]:
    p = f"{OUT}/report/{f}"
    if os.path.exists(p):
        print(f); display(Image(p, width=900))

## 4. Files for the public repository / Zenodo (Data Availability)
Model checkpoints are larger than GitHub's 100 MB file limit when bundled, so deposit `release_bundle.zip`
on Zenodo (or as a GitHub Release asset) and cite its DOI in the manuscript.

In [ ]:
import zipfile, numpy as np, cv2, csv
B = "/kaggle/working/release_bundle.zip"
meta = list(csv.DictReader(open(f"{OUT}/cache/meta.csv")))
Y = np.load(f"{OUT}/cache/Y.npy")
from lcseg.utils import unpack
with zipfile.ZipFile(B, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["groups/groups.csv", "groups/group_stats.json", "cache/prepare.json", "generator/generator.json",
              "bench/bench.json", "report/results.json", "report/results.md"]:
        if os.path.exists(f"{OUT}/{f}"):
            z.write(f"{OUT}/{f}", f)
    for f in glob.glob(f"{OUT}/report/*.png"):
        z.write(f, "report/" + os.path.basename(f))
    for r in glob.glob(f"{OUT}/runs/*"):
        for f in ["model.keras", "run.json", "eval.npz", "eval_time.json"]:
            if os.path.exists(f"{r}/{f}"):
                z.write(f"{r}/{f}", f"runs/{os.path.basename(r)}/{f}")
    for m in meta:                              # generated reference masks as PNG (0/255)
        if m["cls"] in ("lung_aca", "lung_scc", "colon_aca"):
            ok, buf = cv2.imencode(".png", unpack(Y[int(m["idx"])]) * 255)
            z.writestr(f"masks/{m['cls']}/{os.path.splitext(m['file'])[0]}.png", buf.tobytes())
print(B, round(os.path.getsize(B) / 1e6, 1), "MB")